# 1. Filter Massive OSM Dataset


In [1]:
from pathlib import Path

import geopandas as gpd
from shapely.geometry import box


def filter_lta_layer(shapefile_path, output_path, bbox_coords):
    print(f"🔄 Processing {shapefile_path}...")

    # 1. Read the LTA Shapefile
    gdf = gpd.read_file(shapefile_path)

    # 2. Match Singapore's SVY21 coordinate system (EPSG:3414)
    # to standard GPS coordinates (WGS84 - EPSG:4326) if necessary
    if gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    # 3. Create a cropping bounding box around your neighborhood
    min_lon, min_lat, max_lon, max_lat = bbox_coords
    neighborhood_box = box(min_lon, min_lat, max_lon, max_lat)

    # 4. Filter data intersecting this box
    filtered_gdf = gdf[gdf.geometry.intersects(neighborhood_box)]

    # 5. Export back as a clean, localized Shapefile bundle
    filtered_gdf.to_file(output_path)
    print(
        f"✅ Saved localized layer to {output_path} ({len(filtered_gdf)} features found)"
    )


# Get list of shapefiles in raw_data folder
# Loop all folders in GEOSPATIAL folder, get only the string right before underscore to be the shapefile names
OSM_DIR = Path("malaysia-singapore-brunei-260706-free.shp")
OSM_CLEMENTI_DIR = Path("OSM_CLEMENTI")
# Loop subfolders only (skip .zip files)
shapefile_names = []

# --- CONFIGURATION FOR YOUR HDB ESTATE ---
# Example Bounding Box for Central Toa Payoh Area [Min Lon, Min Lat, Max Lon, Max Lat]
CLEMENTI_BBOX = [103.751106, 1.296920, 103.789859, 1.324078]

shp_files = list(OSM_DIR.glob("*.shp"))
for layer in shp_files:
    shapefile_names.append(layer.stem)

# Run the filter for your downloaded layers (Update names according to your extracted file filenames)
for filename in shapefile_names:
    filter_lta_layer(
        str(OSM_DIR / f"{filename}.shp"),
        str(OSM_CLEMENTI_DIR / f"{filename}.shp"),
        CLEMENTI_BBOX,
    )

# Run the filter for your downloaded layers (Update names according to your extracted file filenames)
# filter_lta_layer("raw_data/Footpath.shp", "clementi_footpaths.shp", CLEMENTI_BBOX)
# filter_lta_layer("raw_data/KerbLine.shp", "clementi_kerblines.shp", CLEMENTI_BBOX)

🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_landuse_a_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI/gis_osm_landuse_a_free_1.shp (476 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_natural_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI/gis_osm_natural_free_1.shp (25 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_buildings_a_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI/gis_osm_buildings_a_free_1.shp (4347 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_waterways_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI/gis_osm_waterways_free_1.shp (77 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_pofw_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI/gis_osm_pofw_free_1.shp (2 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_water_a_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI/gis

In [2]:
print(shapefile_names)
shapefile_names = [
    "gis_osm_landuse_a_free_1",
    "gis_osm_natural_free_1",
    "gis_osm_buildings_a_free_1",
    "gis_osm_waterways_free_1",
    "gis_osm_pofw_free_1",
    "gis_osm_water_a_free_1",
    "gis_osm_transport_free_1",
    "gis_osm_places_a_free_1",
    "gis_osm_natural_a_free_1",
    "gis_osm_places_free_1",
    "gis_osm_pofw_a_free_1",
    "gis_osm_pois_free_1",
    "gis_osm_traffic_a_free_1",
    "gis_osm_traffic_free_1",
    "gis_osm_railways_free_1",
    "gis_osm_transport_a_free_1",
    "gis_osm_pois_a_free_1",
    "gis_osm_adminareas_a_free_1",
    "gis_osm_roads_free_1",
    "gis_osm_protected_areas_a_free_1",
]

['gis_osm_landuse_a_free_1', 'gis_osm_natural_free_1', 'gis_osm_buildings_a_free_1', 'gis_osm_waterways_free_1', 'gis_osm_pofw_free_1', 'gis_osm_water_a_free_1', 'gis_osm_transport_free_1', 'gis_osm_places_a_free_1', 'gis_osm_natural_a_free_1', 'gis_osm_places_free_1', 'gis_osm_pofw_a_free_1', 'gis_osm_pois_free_1', 'gis_osm_traffic_a_free_1', 'gis_osm_traffic_free_1', 'gis_osm_railways_free_1', 'gis_osm_transport_a_free_1', 'gis_osm_pois_a_free_1', 'gis_osm_adminareas_a_free_1', 'gis_osm_roads_free_1', 'gis_osm_protected_areas_a_free_1']


# 2. Convert OSM to NetworkX Graph


In [3]:
neighbourhood_gpd = gpd.read_file(str(OSM_CLEMENTI_DIR / "gis_osm_roads_free_1.shp"))

In [4]:
import networkx as nx

G = nx.Graph()

for idx, row in neighbourhood_gpd.iterrows():
    coords = list(row.geometry.coords)
    # Connect points in the linestring as edges in the graph
    for i in range(len(coords) - 1):
        u = coords[i]
        v = coords[i + 1]

        # Calculate flat geographical distance as initial weight
        dist = row.geometry.length

        G.add_edge(
            u, v, weight=dist, distance=dist, osm_id=row["osm_id"], fclass=row["fclass"]
        )

# 3. Inject and Snap LTA DataMall Nodes


In [5]:
from scipy.spatial import KDTree

## Add Bus Stops


In [6]:
CLEMENTI_DIR = Path("CLEMENTI")
# 1. Extract all node coordinates from your NetworkX graph
graph_nodes = list(G.nodes())
tree = KDTree(graph_nodes)

# 2. Load your LTA DataMall shapefile (e.g., BusStop)
lta_bus_stops = gpd.read_file(str(CLEMENTI_DIR / "BusStop_Clementi.shp"))

# 3. Snap each bus stop point to the nearest OSM node
for idx, row in lta_bus_stops.iterrows():
    bus_coord = (row.geometry.x, row.geometry.y)

    # Find closest node in your base graph
    distance, min_idx = tree.query(bus_coord)
    closest_node = graph_nodes[min_idx]

    # Enrich that node!
    G.nodes[closest_node]["is_bus_stop"] = True
    G.nodes[closest_node]["bus_stop_name"] = row.get("LOC_DESC", "Unknown")
    G.nodes[closest_node]["bus_stop_id"] = row.get("BUS_STOP_N", "Unknown")

## Add MRT Stations


In [7]:
# is a polygon
lta_mrt_station = gpd.read_file(
    str(CLEMENTI_DIR / "RapidTransitSystemStation_Clementi.shp")
)

print(lta_mrt_station)
print(lta_mrt_station["geometry"].head())
print(lta_mrt_station.columns)

   TYP_CD STN_NAM        ATTACHEMEN   SHAPE_AREA   SHAPE_LEN TYP_CD_DES  \
0       0    None               NaN  4684.677684  405.638249        MRT   
1       0    None  CC23_ONH STN.zip  5349.313823  453.125437        MRT   
2       0    None               NaN  5517.426086  588.824659        MRT   
3       0    None               NaN  2884.848941  334.622130        MRT   

                STN_NAM_DE                                           geometry  
0        DOVER MRT STATION  POLYGON ((103.77929 1.31089, 103.77929 1.31089...  
1    ONE-NORTH MRT STATION  POLYGON ((103.78757 1.30016, 103.78761 1.30023...  
2  BUONA VISTA MRT STATION  POLYGON ((103.78943 1.30742, 103.79075 1.3073,...  
3     CLEMENTI MRT STATION  POLYGON ((103.76543 1.31444, 103.76542 1.31443...  
0    POLYGON ((103.77929 1.31089, 103.77929 1.31089...
1    POLYGON ((103.78757 1.30016, 103.78761 1.30023...
2    POLYGON ((103.78943 1.30742, 103.79075 1.3073,...
3    POLYGON ((103.76543 1.31444, 103.76542 1.31443...
Name

In [11]:
# Extract Clementi for Train Station Exit
def filter_lta_layer(shapefile_path, output_path, bbox_coords):
    print(f"🔄 Processing {shapefile_path}...")

    # 1. Read the LTA Shapefile
    gdf = gpd.read_file(shapefile_path)

    # 2. Match Singapore's SVY21 coordinate system (EPSG:3414)
    # to standard GPS coordinates (WGS84 - EPSG:4326) if necessary
    if gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    # 3. Create a cropping bounding box around your neighborhood
    min_lon, min_lat, max_lon, max_lat = bbox_coords
    neighborhood_box = box(min_lon, min_lat, max_lon, max_lat)

    # 4. Filter data intersecting this box
    filtered_gdf = gdf[gdf.geometry.intersects(neighborhood_box)]

    # 5. Export back as a clean, localized Shapefile bundle
    filtered_gdf.to_file(output_path)
    print(
        f"✅ Saved localized layer to {output_path} ({len(filtered_gdf)} features found)"
    )


layer = Path("GEOSPATIAL") / "TrainStationExit_Jul2026" / "Train_Station_Exit_Layer.shp"
filter_lta_layer(
    str(layer),
    str(CLEMENTI_DIR / "TrainStationExit_Clementi.shp"),
    CLEMENTI_BBOX,
)

🔄 Processing GEOSPATIAL/TrainStationExit_Jul2026/Train_Station_Exit_Layer.shp...
✅ Saved localized layer to CLEMENTI/TrainStationExit_Clementi.shp (10 features found)


In [12]:
# is a polygon
mrt_station_exit = gpd.read_file(str(CLEMENTI_DIR / "TrainStationExit_Clementi.shp"))

print(mrt_station_exit)
print(mrt_station_exit["geometry"].head())
print(mrt_station_exit.columns)

                stn_name exit_code                   geometry
0  ONE-NORTH MRT STATION    Exit A  POINT (103.78691 1.29903)
1  ONE-NORTH MRT STATION    Exit B  POINT (103.78793 1.30036)
2      DOVER MRT STATION    Exit A   POINT (103.7783 1.31142)
3      DOVER MRT STATION    Exit B  POINT (103.77846 1.31169)
4  ONE-NORTH MRT STATION    Exit C  POINT (103.78757 1.29971)
5   CLEMENTI MRT STATION    Exit A  POINT (103.76498 1.31537)
6   CLEMENTI MRT STATION    Exit B  POINT (103.76519 1.31547)
7   CLEMENTI MRT STATION    Exit D  POINT (103.76557 1.31408)
8   CLEMENTI MRT STATION    Exit C  POINT (103.76575 1.31416)
9  ONE-NORTH MRT STATION    Exit D  POINT (103.78778 1.30007)
0    POINT (103.78691 1.29903)
1    POINT (103.78793 1.30036)
2     POINT (103.7783 1.31142)
3    POINT (103.77846 1.31169)
4    POINT (103.78757 1.29971)
Name: geometry, dtype: geometry
Index(['stn_name', 'exit_code', 'geometry'], dtype='str')


In [13]:
import pandas as pd
import geopandas as gpd
import networkx as nx
from scipy.spatial import KDTree

# Ensure G is your base network graph built from 'gis_osm_roads_free_1'
# G = nx.Graph(...)

# 1. Extract all node coordinates from your existing NetworkX graph
graph_nodes = list(G.nodes())  # Array of (longitude, latitude) tuples
graph_tree = KDTree(graph_nodes)

print(f"Loaded base graph with {len(graph_nodes)} pedestrian nodes.")

# 2. Iterate through each physical MRT exit point
for idx, row in mrt_station_exit.iterrows():
    station_name = row["stn_name"]  # e.g., "CLEMENTI MRT STATION"
    exit_code = row["exit_code"]  # e.g., "Exit A"

    # Extract the raw (lng, lat) from the POINT geometry
    exit_coord = (row.geometry.x, row.geometry.y)

    # 3. Query the KDTree to find the absolute closest node in your walking graph
    # distance is in degrees; min_idx is the index of the closest node in graph_nodes
    distance, min_idx = graph_tree.query(exit_coord)
    closest_node = graph_nodes[min_idx]

    # 4. Enrich that specific network node with attributes
    G.nodes[closest_node]["is_transit_hub"] = True
    G.nodes[closest_node]["node_type"] = "mrt_exit"
    G.nodes[closest_node]["station_name"] = station_name
    G.nodes[closest_node]["exit_code"] = exit_code

    # Let's add a safe connection attribute to the node
    if "assigned_exits" not in G.nodes[closest_node]:
        G.nodes[closest_node]["assigned_exits"] = []

    G.nodes[closest_node]["assigned_exits"].append(f"{station_name} - {exit_code}")

    print(
        f"⚓ Snapped {station_name} ({exit_code}) to Graph Node {closest_node} (Dist: {distance:.5f} deg)"
    )

print("\n✅ MRT Exits successfully integrated into the Network Graph!")

Loaded base graph with 27948 pedestrian nodes.
⚓ Snapped ONE-NORTH MRT STATION (Exit A) to Graph Node (103.7868787, 1.2990096) (Dist: 0.00003 deg)
⚓ Snapped ONE-NORTH MRT STATION (Exit B) to Graph Node (103.7877482, 1.3002732) (Dist: 0.00020 deg)
⚓ Snapped DOVER MRT STATION (Exit A) to Graph Node (103.7783048, 1.3114656) (Dist: 0.00005 deg)
⚓ Snapped DOVER MRT STATION (Exit B) to Graph Node (103.7784313, 1.3116727) (Dist: 0.00004 deg)
⚓ Snapped ONE-NORTH MRT STATION (Exit C) to Graph Node (103.7875814, 1.2998009) (Dist: 0.00009 deg)
⚓ Snapped CLEMENTI MRT STATION (Exit A) to Graph Node (103.765008, 1.3153734) (Dist: 0.00003 deg)
⚓ Snapped CLEMENTI MRT STATION (Exit B) to Graph Node (103.7651836, 1.3154475) (Dist: 0.00002 deg)
⚓ Snapped CLEMENTI MRT STATION (Exit D) to Graph Node (103.7655577, 1.3140042) (Dist: 0.00008 deg)
⚓ Snapped CLEMENTI MRT STATION (Exit C) to Graph Node (103.7656688, 1.3141081) (Dist: 0.00010 deg)
⚓ Snapped ONE-NORTH MRT STATION (Exit D) to Graph Node (103.787783

## Add Passenger Pickup Bay


In [ ]:
passenger_pickup_bay = gpd.read_file(
    str(CLEMENTI_DIR / "PassengerPickupBay_Clementi.shp")
)

print(passenger_pickup_bay)
print(passenger_pickup_bay["geometry"].head())
print(passenger_pickup_bay.columns)

   OBJECTID                                          geometry
0        49  LINESTRING (103.7778 1.31167, 103.77791 1.31159)
1       103  LINESTRING (103.7793 1.31128, 103.77942 1.31119)
0    LINESTRING (103.7778 1.31167, 103.77791 1.31159)
1    LINESTRING (103.7793 1.31128, 103.77942 1.31119)
Name: geometry, dtype: geometry
Index(['OBJECTID', 'geometry'], dtype='str')


In [ ]:
import geopandas as gpd
import networkx as nx
from scipy.spatial import KDTree
from shapely.geometry import Point

# Ensure G is your base network graph built from 'gis_osm_roads_free_1'
# G = nx.Graph(...)

# 1. Extract all node coordinates from your walking graph
graph_nodes = list(G.nodes())  # Array of (longitude, latitude) tuples
graph_tree = KDTree(graph_nodes)

print(f"Loaded base graph with {len(graph_nodes)} pedestrian nodes.")

# 2. Iterate through each Passenger Pickup Bay LineString
for idx, row in passenger_pickup_bay.iterrows():
    object_id = row["OBJECTID"]
    geom = row.geometry

    # Extract the first point (Start) and last point (End) of the line string
    start_coord = (geom.coords[0][0], geom.coords[0][1])  # (lng, lat)
    end_coord = (geom.coords[-1][0], geom.coords[-1][1])  # (lng, lat)

    # 3. Snap both the start and end coordinates to the nearest pedestrian walkway nodes
    for coord_type, coord in [("Start", start_coord), ("End", end_coord)]:
        distance, min_idx = graph_tree.query(coord)
        closest_node = graph_nodes[min_idx]

        # 4. Enrich the snapped pedestrian network node
        G.nodes[closest_node]["is_pickup_bay"] = True
        G.nodes[closest_node]["node_type"] = "passenger_pickup"
        G.nodes[closest_node]["pickup_id"] = f"Bay_{object_id}"

        # Initialize an tracking list for multiple assignments if paths overlap
        if "assigned_bays" not in G.nodes[closest_node]:
            G.nodes[closest_node]["assigned_bays"] = []
        G.nodes[closest_node]["assigned_bays"].append(f"Bay {object_id} ({coord_type})")

        print(
            f"🚗 Snapped Pickup Bay {object_id} {coord_type} point to Graph Node {closest_node} (Dist: {distance:.5f} deg)"
        )

print("\n✅ Passenger Pickup Bays successfully integrated into the Network Graph!")

Loaded base graph with 27948 pedestrian nodes.
🚗 Snapped Pickup Bay 49 Start point to Graph Node (103.7778774, 1.3116705) (Dist: 0.00008 deg)
🚗 Snapped Pickup Bay 49 End point to Graph Node (103.7778774, 1.3116705) (Dist: 0.00009 deg)
🚗 Snapped Pickup Bay 103 Start point to Graph Node (103.779207, 1.3114106) (Dist: 0.00016 deg)
🚗 Snapped Pickup Bay 103 End point to Graph Node (103.7794679, 1.3112215) (Dist: 0.00006 deg)

✅ Passenger Pickup Bays successfully integrated into the Network Graph!


## Add Taxi Stand


In [18]:
taxi_stand = gpd.read_file(str(CLEMENTI_DIR / "TaxiStop_Clementi.shp"))

print(taxi_stand)
print(taxi_stand["geometry"].head())
print(taxi_stand.columns)

  TYPE_CD LOC_NUM                                LOC_DESC  TYPE_CD_DE  \
0    None     H01                          OUTSIDE BLK 19  TAXI STAND   
1    None     J06      DOVER MRT STN (TOWARDS QUEENSTOWN)  TAXI STAND   
2    None     J07  CLEMENTI MRT STN (TOWARDS JURONG EAST)  TAXI STAND   
3    None     NaN                                     NaN  TAXI STAND   
4    None     F60                                THE STAR  TAXI STAND   
5    None     J05         DOVER MRT STN (TOWARD CLEMENTI)  TAXI STAND   
6    None     H04    DRIVEWAY OUTSIDE COLD STORAGE JELITA  TAXI STAND   

                    geometry  
0  POINT (103.78797 1.31178)  
1   POINT (103.77916 1.3114)  
2  POINT (103.76548 1.31412)  
3  POINT (103.78372 1.29829)  
4  POINT (103.78825 1.30645)  
5  POINT (103.77793 1.31156)  
6  POINT (103.78595 1.31775)  
0    POINT (103.78797 1.31178)
1     POINT (103.77916 1.3114)
2    POINT (103.76548 1.31412)
3    POINT (103.78372 1.29829)
4    POINT (103.78825 1.30645)
Name: geomet

In [19]:
graph_nodes = list(G.nodes())
tree = KDTree(graph_nodes)

# Snap each taxi stand point to the nearest OSM node
for idx, row in taxi_stand.iterrows():
    taxi_stand_coord = (row.geometry.x, row.geometry.y)

    # Find closest node in your base graph
    distance, min_idx = tree.query(bus_coord)
    closest_node = graph_nodes[min_idx]

    # Enrich that node!
    G.nodes[closest_node]["is_taxi_stand"] = True
    G.nodes[closest_node]["node_type"] = "taxi_stand"
    G.nodes[closest_node]["taxi_stand_name"] = row.get("LOC_DESC", "Unknown")
    G.nodes[closest_node]["taxi_stand_id"] = row.get("LOC_NUM", "Unknown")

## Add Footpath


In [20]:
footpath = gpd.read_file(str(CLEMENTI_DIR / "Footpath_Clementi.shp"))

print(footpath)
print(footpath["geometry"].head())
print(footpath.columns)

      OBJECTID                                           geometry
0          133  LINESTRING (103.76561 1.29795, 103.76562 1.29792)
1        79296  LINESTRING (103.76536 1.29847, 103.76549 1.298...
2          247   LINESTRING (103.78689 1.31766, 103.78699 1.3176)
3        18804  LINESTRING (103.7671 1.32001, 103.76711 1.3199...
4        99110  LINESTRING (103.76416 1.31547, 103.76416 1.315...
...        ...                                                ...
3955     77570  MULTILINESTRING ((103.78957 1.30155, 103.78956...
3956     33941  LINESTRING (103.78703 1.32052, 103.78703 1.320...
3957     34249  LINESTRING (103.78692 1.32049, 103.78689 1.32063)
3958     96740  LINESTRING (103.78693 1.32043, 103.78695 1.32036)
3959     96741  LINESTRING (103.78696 1.32033, 103.78698 1.320...

[3960 rows x 2 columns]
0    LINESTRING (103.76561 1.29795, 103.76562 1.29792)
1    LINESTRING (103.76536 1.29847, 103.76549 1.298...
2     LINESTRING (103.78689 1.31766, 103.78699 1.3176)
3    LINESTRING (1

In [ ]:
import geopandas as gpd
import networkx as nx
from scipy.spatial import KDTree
from shapely.geometry import LineString

# G = your existing base graph built from 'gis_osm_roads_free_1'
# Initialize all existing base graph edges to default False for footpaths
for u, v in G.edges():
    G.edges[u, v]["is_footpath"] = False
    G.edges[u, v]["source"] = "OSM_Base"

# 1. Extract all node coordinates from your current OSM graph
graph_nodes = list(G.nodes())  # List of (longitude, latitude) tuples
graph_tree = KDTree(graph_nodes)

# Define a tight matching threshold (e.g., ~5 meters)
# In WGS84 degrees, 0.00005 is roughly 5.5 meters
MATCH_THRESHOLD_DEG = 0.00005

edges_marked = 0
edges_created = 0

print(f"Aligning {len(footpath)} LTA footprint features with the OSM base graph...")

# 2. Loop through each LTA Footpath line
for idx, row in footpath.iterrows():
    geom = row.geometry
    obj_id = row["OBJECTID"]

    # Handle both LineString and MultiLineString structures smoothly
    if geom.geom_type == "LineString":
        lines = [geom]
    elif geom.geom_type == "MultiLineString":
        lines = list(geom.geoms)
    else:
        continue

    for line in lines:
        # Get start and end coordinates of this LTA footpath segment
        start_coord = (line.coords[0][0], line.coords[0][1])
        end_coord = (line.coords[-1][0], line.coords[-1][1])

        # 3. Find the closest corresponding nodes in the OSM graph
        dist_start, idx_start = graph_tree.query(start_coord)
        dist_end, idx_end = graph_tree.query(end_coord)

        # Check if both endpoints fall safely within our spatial matching threshold
        if dist_start <= MATCH_THRESHOLD_DEG and dist_end <= MATCH_THRESHOLD_DEG:
            node_u = graph_nodes[idx_start]
            node_v = graph_nodes[idx_end]

            if node_u == node_v:
                continue  # Skip if it snaps to the exact same node

            # Scenario A: The edge already exists in OSM -> Mark it!
            if G.has_edge(node_u, node_v):
                G.edges[node_u, node_v]["is_footpath"] = True
                G.edges[node_u, node_v]["lta_object_id"] = obj_id
                G.edges[node_u, node_v]["source"] = "OSM_LTA_Matched"
                edges_marked += 1

            # Scenario B: The pathway route exists physically but is missing an edge -> Create it!
            else:
                calc_dist = LineString([node_u, node_v]).length
                G.add_edge(
                    node_u,
                    node_v,
                    weight=calc_dist,
                    distance=calc_dist,
                    is_footpath=True,
                    lta_object_id=obj_id,
                    source="LTA_Injected_Edge",
                )
                edges_created += 1

print(f"\n⚡ Alignment Finished!")
print(f"🔹 Existing OSM edges marked as official footpaths: {edges_marked}")
print(f"🔸 Missing edges created and added to base graph: {edges_created}")

Aligning 3960 LTA footprint features with the OSM base graph...

⚡ Alignment Finished!
🔹 Existing OSM edges marked as official footpaths: 288
🔸 Missing edges created and added to base graph: 555


## Add Pedestrian Overhead Bridge/Underpass


## Add Road Crossing


## Add Cyclying Path


## Add Covered Linkway


## Add Traffic Light


## Add Traffic Sign


# 4. Add some necessary Buildings from URA like HDB, Malls, etc.


### Save your enriched graph


In [14]:
import json
import networkx as nx

# 4. Save your enriched graph using standard XML GraphML
nx.write_graphml(G, "clementi_graph.graphml")

NetworkXError: GraphML writer does not support <class 'list'> as data values.

### Load the Graph Back


In [ ]:
# 5. Load your graph back in
G = nx.read_graphml("clementi_graph.graphml")

# 5. Export Graph to JSON


In [ ]:
import json
import networkx as nx
from shapely.geometry import Point, LineString, mapping


def export_graph_to_geojson(G, output_filename="singapore_3d_sim.geojson"):
    features = []

    # ----------------------------------------------------
    # 1. Export Graph Nodes (Transit & Obstacle Markers)
    # ----------------------------------------------------
    for node, data in G.nodes(data=True):
        # Coordinates from the node key tuple (lng, lat)
        lng, lat = node

        # Clean up any potential list types into safe JSON strings
        properties = {}
        for key, value in data.items():
            if isinstance(value, list):
                properties[key] = ", ".join(map(str, value))
            else:
                properties[key] = value

        # Inject standard properties required for the frontend
        properties["lng"] = lng
        properties["lat"] = lat

        node_feature = {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": properties,
        }
        features.append(node_feature)

    # ----------------------------------------------------
    # 2. Export Graph Edges (Walkway & Road Connections)
    # ----------------------------------------------------
    for u, v, data in G.edges(data=True):
        # Edge geometry line connecting the two nodes
        line_geom = LineString([u, v])

        properties = {
            "is_footpath": data.get("is_footpath", False),
            "weight": data.get("weight", 0.0),
            "distance": data.get("distance", 0.0),
            "source": data.get("source", "OSM_Base"),
        }

        edge_feature = {
            "type": "Feature",
            "geometry": mapping(line_geom),
            "properties": properties,
        }
        features.append(edge_feature)

    # Compile final GeoJSON Feature Collection
    geojson_payload = {"type": "FeatureCollection", "features": features}

    # Save directly to your local workspace
    with open(output_filename, "w") as f:
        json.dump(geojson_payload, f, indent=2)

    print(
        f"✅ Successfully compiled and exported {len(features)} spatial elements to {output_filename}"
    )


# Execute the exporter at the end of your Jupyter cells
export_graph_to_geojson(G)

✅ Successfully compiled and exported 59727 spatial elements to singapore_3d_sim.geojson
